# 18.7 语义缓存 (Semantic Caching)

> 🕐 预估学习时间：30分钟

对语义相近的请求复用历史回答或 KV/检索结果，可显著降低成本与延迟。与精确字符串缓存不同，语义缓存用嵌入相似度判定命中。

本节涵盖：
- 精确缓存 vs 语义缓存
- 相似度阈值与风险
- 缓存分层（答案缓存 / 检索缓存 / 前缀 KV）
- 失效、租户隔离与评测


## 1. 语义命中判定

对查询做嵌入，若与缓存键的余弦相似度 ≥ τ 则命中。τ 过高降命中率，过低导致错误复用。


In [ ]:
import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)


class SemanticCache:
    def __init__(self, dim=64, threshold=0.88, max_size=100):
        self.threshold = threshold
        self.max_size = max_size
        self.keys = []    # tensors
        self.values = []  # payloads
        self.hits = 0
        self.misses = 0

    def _sim(self, q, k):
        return F.cosine_similarity(q.unsqueeze(0), k.unsqueeze(0)).item()

    def get(self, q_emb):
        best_i, best_s = -1, -1.0
        for i, k in enumerate(self.keys):
            s = self._sim(q_emb, k)
            if s > best_s:
                best_i, best_s = i, s
        if best_s >= self.threshold:
            self.hits += 1
            return self.values[best_i], best_s
        self.misses += 1
        return None, best_s

    def put(self, q_emb, value):
        if len(self.keys) >= self.max_size:
            self.keys.pop(0)
            self.values.pop(0)
        self.keys.append(q_emb.detach().cpu())
        self.values.append(value)


cache = SemanticCache(threshold=0.9)
base = F.normalize(torch.randn(64), dim=0)
cache.put(base, 'answer-about-lora')

# near duplicate
q_near = F.normalize(base + 0.01 * torch.randn(64), dim=0)
q_far = F.normalize(torch.randn(64), dim=0)
print('=== Semantic Cache Lookup ===')
print('near:', cache.get(q_near))
print('far:', cache.get(q_far))
print(f'hit_rate={cache.hits/(cache.hits+cache.misses):.2f}')
print(f'Key: Threshold τ is a safety-critical knob; tune on paraphrase/adversarial eval sets.')


## 2. 分层缓存策略

| 层 | 缓存对象 | 命中收益 | 风险 |
|----|---------|---------|------|
| L0 精确 | 规范化后的字符串 | 极低风险 | 命中率低 |
| L1 语义答案 | 最终回复 | 高 | 事实过期/个性化串味 |
| L2 检索结果 | Top-K 文档 | 中 | 语料更新需失效 |
| L3 前缀 KV | 系统提示 KV | 高 | 与引擎耦合 |

个性化、含工具结果或强时效问题应 bypass 或短 TTL。


In [ ]:
class LayeredCache:
    def __init__(self):
        self.exact = {}
        self.semantic = SemanticCache(threshold=0.92)
        self.retrieval = SemanticCache(threshold=0.9)

    def normalize(self, q: str) -> str:
        return ' '.join(q.lower().strip().split())

    def answer(self, query: str, embed_fn, generate_fn, retrieve_fn):
        key = self.normalize(query)
        if key in self.exact:
            return self.exact[key], 'exact'
        emb = embed_fn(query)
        hit, score = self.semantic.get(emb)
        if hit is not None:
            return hit, f'semantic:{score:.3f}'
        docs, dtype = None, None
        d_hit, d_score = self.retrieval.get(emb)
        if d_hit is not None:
            docs, dtype = d_hit, f'retrieval_cache:{d_score:.3f}'
        else:
            docs = retrieve_fn(query)
            self.retrieval.put(emb, docs)
            dtype = 'retrieval_fresh'
        ans = generate_fn(query, docs)
        self.exact[key] = ans
        self.semantic.put(emb, ans)
        return ans, dtype


def embed_fn(q):
    # Topic centroid + small query noise: paraphrases land nearby
    qn = ' '.join(q.lower().split())
    if 'lora' in qn:
        base = torch.tensor([1.0] + [0.0] * 63)
    elif 'kv' in qn or 'cache' in qn or 'caching' in qn:
        base = torch.tensor([0.0, 1.0] + [0.0] * 62)
    else:
        g = torch.Generator().manual_seed(sum(map(ord, qn)) % (2**31))
        base = torch.randn(64, generator=g)
    g2 = torch.Generator().manual_seed(sum(map(ord, qn)) % (2**31))
    noise = 0.05 * torch.randn(64, generator=g2)
    return F.normalize(base + noise, dim=0)

def retrieve_fn(q):
    return [f'doc-for:{q[:12]}']

def generate_fn(q, docs):
    return f'ans({q[:16]}) relying on {docs[0]}'

lc = LayeredCache()
print('=== Layered Cache ===')
a1, s1 = lc.answer('What is LoRA?', embed_fn, generate_fn, retrieve_fn)
a2, s2 = lc.answer('what is lora?', embed_fn, generate_fn, retrieve_fn)
a3, s3 = lc.answer('What is LoRA fine-tuning?', embed_fn, generate_fn, retrieve_fn)
print(s1, '->', a1)
print(s2, '->', a2)
print(s3, '->', a3)
print(f'\nKey: Exact cache catches normalization duplicates; semantic cache catches paraphrases; retrieval cache saves vector DB cost.')


## 3. 评测与失效

必须监控：命中率、错误复用率、TTL 命中贡献、每租户隔离。

失效触发：知识库更新、政策变更、模型版本切换、安全事件。


In [ ]:
def evaluate_threshold(paraphrase_pairs, unrelated_pairs, embed_fn, thresholds):
    print(f'{"τ":>6} {"para_hit":>10} {"false_hit":>10}')
    for t in thresholds:
        c = SemanticCache(threshold=t)
        # populate with left side
        for a, _ in paraphrase_pairs + unrelated_pairs:
            c.put(embed_fn(a), f'ans:{a}')
        # reset counters
        c.hits = c.misses = 0
        para_hit = 0
        for a, b in paraphrase_pairs:
            # fresh cache per lookup population already has a
            val, _ = c.get(embed_fn(b))
            para_hit += int(val is not None)
        false_hit = 0
        # rebuild for fair false-hit measurement
        c2 = SemanticCache(threshold=t)
        for a, _ in unrelated_pairs:
            c2.put(embed_fn(a), f'ans:{a}')
        for a, b in unrelated_pairs:
            val, _ = c2.get(embed_fn(b))
            false_hit += int(val is not None)
        print(f'{t:>6.2f} {para_hit/len(paraphrase_pairs):>10.2f} {false_hit/len(unrelated_pairs):>10.2f}')


pairs_para = [('lora tuning', 'lora fine tuning'), ('kv cache', 'kv caching')]
pairs_unrel = [('lora tuning', 'pancake recipe'), ('kv cache', 'stock market')]
evaluate_threshold(pairs_para, pairs_unrel, embed_fn, [0.7, 0.8, 0.9, 0.95])
print(f'\nKey: Choose τ on the Pareto frontier of paraphrase hit rate vs false-hit rate.')


## 课后思考题

1. 哪些请求类型绝对不应语义缓存？如何在网关层识别？
2. 多租户下语义缓存如何防止跨租户答案泄漏？
3. 语义缓存与前缀 KV 缓存、CDN 缓存在职责上如何划分？
4. 知识库更新后如何做细粒度失效而不是清空全站缓存？

---
> 本节涵盖了18.7 语义缓存的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
